In [1]:
import vrep 
import sys
import time 
import numpy as np
from tank import *

In [3]:
vrep.simxFinish(-1) # closes all opened connections, in case any prevoius wasnt finished
clientID=vrep.simxStart('127.0.0.1',19999,True,True,5000,5) # start a connection

if clientID!=-1:
    print ("Connected to remote API server")
else:
    print("Not connected to remote API server")
    sys.exit("Could not connect")

#create instance of Tank
tank=Tank(clientID)

# get handle to proximity sensor
err_code,ps_handle = vrep.simxGetObjectHandle(clientID,"Proximity_sensor", vrep.simx_opmode_blocking)

import skfuzzy as fuzz
from skfuzzy import control as ctrl

def create_fuzzy_controller(mode='lagodny'):
    distance = ctrl.Antecedent(np.arange(0, 10.01, 0.01), 'distance')
    speed = ctrl.Consequent(np.arange(0, 30.01, 0.01), 'speed')
    speed0 = 0
    
    if mode == 'lagodny':
        distance['stop'] = fuzz.trimf(distance.universe, [0.0, 0.0, 0.3]) 
        distance['very_close'] = fuzz.trimf(distance.universe, [0.15, 0.4, 0.6])
        distance['close'] = fuzz.trimf(distance.universe, [0.5, 0.7, 1.2]) 
        distance['mid'] = fuzz.trimf(distance.universe, [1.0, 2.5, 5.0])
        distance['far'] = fuzz.trimf(distance.universe, [4.0, 7.0, 10.0])
        distance['very_far'] = fuzz.trimf(distance.universe, [8.0, 10.0, 10.0])

        speed['stop'] = fuzz.trimf(speed.universe, [0, 0, 0.1])  
        speed['very_slow'] = fuzz.trimf(speed.universe, [0.05, 0.5, 2.0]) 
        speed['slow'] = fuzz.trimf(speed.universe, [1.5, 4.0, 8.0])
        speed['mid'] = fuzz.trimf(speed.universe, [6.0, 12.0, 18.0])
        speed['fast'] = fuzz.trimf(speed.universe, [16.0, 22.0, 30.0])

        rules = [
            ctrl.Rule(distance['stop'], speed['stop']),
            ctrl.Rule(distance['very_close'], speed['very_slow']),
            ctrl.Rule(distance['close'], speed['slow']),
            ctrl.Rule(distance['mid'], speed['mid']),
            ctrl.Rule(distance['far'], speed['fast']),
            ctrl.Rule(distance['very_far'], speed['fast']),
        ]
    
    elif mode == 'agresywny':
        distance['stop'] = fuzz.trimf(distance.universe, [0.0, 0.0, 0.15])  
        distance['mid'] = fuzz.trimf(distance.universe, [0.12, 0.3, 1])  
        distance['far'] = fuzz.trimf(distance.universe, [0.8, 3.0, 10.0])

        speed['stop'] = fuzz.trimf(speed.universe, [0, 0, 0.01])  
        speed['slow'] = fuzz.trimf(speed.universe, [0.02, 1.6, 2.5]) 
        speed['fast'] = fuzz.trimf(speed.universe, [12.0, 22.0, 30.0])

        rules = [
            ctrl.Rule(distance['stop'], speed['stop']),
            ctrl.Rule(distance['mid'], speed['slow']),
            ctrl.Rule(distance['far'], speed['fast']),
        ]

    system = ctrl.ControlSystem(rules)
    sim = ctrl.ControlSystemSimulation(system)

    return sim

_, _, _, _, _ = vrep.simxReadProximitySensor(clientID, ps_handle, vrep.simx_opmode_streaming)

sim = create_fuzzy_controller('agresywny')
start_speed = 30
print(f"Symulacja na prędkości początkowej: {start_speed}")
t = time.time()
counter = 0
dist_stop = 0.5

tank.go()
tank.leftvelocity = start_speed
tank.rightvelocity = start_speed
tank.setVelocity()
print(f"Check velocity: {tank.leftvelocity}")

previous_speed = start_speed
while (time.time() - t) < 10:
    err_code, detectionState, detectedPoint, detectedObjectHandle, detectedSurfaceNormalVector = vrep.simxReadProximitySensor(clientID, ps_handle, vrep.simx_opmode_buffer)
    
    if err_code == vrep.simx_return_ok and detectionState:
        distance = np.linalg.norm(detectedPoint)
    else:
        distance = 10
    
    sim.input['distance'] = distance
    # print(f"Distance: {distance}")
    try:
        sim.compute()
        speed = sim.output['speed']

        # speed = 0.5 * speed + 0.5 * previous_speed
        
        previous_speed = speed
    except Exception as e:
        print(e)
        speed = 0
    tank.go()
    tank.leftvelocity = speed
    tank.rightvelocity = speed
    tank.setVelocity()
        
    if counter % 1 == 0:
        print(f"Distance: {distance:.2f}, Speed (fuzzy): {speed:.2f}, Left velocity: {tank.leftvelocity:.2f}, Right velocity: {tank.rightvelocity:.2f}")
    
    counter += 1
    time.sleep(0.1)

vrep.simxStopSimulation(clientID, vrep.simx_opmode_oneshot) # stop the simulation in vrep

Connected to remote API server
Symulacja na prędkości początkowej: 30
Check velocity: 30
'speed'
Distance: 10.00, Speed (fuzzy): 0.00, Left velocity: 0.00, Right velocity: 0.00
Distance: 7.32, Speed (fuzzy): 21.18, Left velocity: 21.18, Right velocity: 21.18
Distance: 7.32, Speed (fuzzy): 21.18, Left velocity: 21.18, Right velocity: 21.18
Distance: 7.33, Speed (fuzzy): 21.18, Left velocity: 21.18, Right velocity: 21.18
Distance: 7.38, Speed (fuzzy): 21.17, Left velocity: 21.17, Right velocity: 21.17
Distance: 7.46, Speed (fuzzy): 21.17, Left velocity: 21.17, Right velocity: 21.17
Distance: 7.48, Speed (fuzzy): 21.17, Left velocity: 21.17, Right velocity: 21.17
Distance: 7.47, Speed (fuzzy): 21.17, Left velocity: 21.17, Right velocity: 21.17
Distance: 7.41, Speed (fuzzy): 21.17, Left velocity: 21.17, Right velocity: 21.17
Distance: 7.23, Speed (fuzzy): 21.18, Left velocity: 21.18, Right velocity: 21.18
Distance: 7.03, Speed (fuzzy): 21.19, Left velocity: 21.19, Right velocity: 21.19
Dis

1